# Phase 1 · Paper methods on CIFAR-100

Attach **alincijov/cifar-100**, enable Internet and GPU, then run this notebook in a fresh session.
Compare exactly **Original, Retrain, Boundary Shrink, Boundary Expanding**.
Forget class **4 = beaver**; retain the other 99 fine classes.

This transfers the author's implementation to CIFAR-100; it is not an exact reproduction
of every experiment in the paper. Input/FGSM normalization is `matched` (mean/std 0.5).
Keep the current training settings: 30 epochs / LR 0.1 / batch 64; unlearning 10 epochs / LR 1e-5.
Plain Shrink uses the author's BN update behavior. Distance weighting, KL and BN ablations are in Phase 2.

The final cell creates **/kaggle/working/result.zip** with metrics, configuration, figures,
best-training checkpoints and a separate **Phase 1 bundle of the exact four evaluated models**.
Download this archive before starting Phase 2. Evaluation saves the bundle immediately;
rerunning the export cell only archives existing outputs and never trains models.


In [ ]:
# Run this cell first, both locally and on Kaggle.
from pathlib import Path
import importlib
import subprocess
import sys

GITHUB_REPO = "https://github.com/ngvihoa/boundary-unlearning.git"  
GITHUB_REF = "main"  # Branch, tag, or commit SHA. Use a SHA to repeat an experiment.
REPO_DIR = Path("/kaggle/working/boundary-unlearning")

if GITHUB_REPO:
    if not (REPO_DIR / ".git").exists():
        subprocess.run([
            "git", "clone", "--depth", "1", "--no-checkout",
            GITHUB_REPO, str(REPO_DIR),
        ], check=True)
    subprocess.run([
        "git", "-C", str(REPO_DIR), "fetch", "--depth", "1", "origin", GITHUB_REF,
    ], check=True)
    subprocess.run([
        "git", "-C", str(REPO_DIR), "checkout", "--detach", "FETCH_HEAD",
    ], check=True)
    CODE_ROOT = REPO_DIR
else:
    CODE_ROOT = Path.cwd()

if not (CODE_ROOT / "boundary_unlearning" / "__init__.py").is_file():
    raise FileNotFoundError(
        "Set GITHUB_REPO to your GitHub HTTPS URL on Kaggle, "
        "or run this notebook from the local repository root."
    )

sys.path.insert(0, str(CODE_ROOT))
importlib.invalidate_caches()
print("Code root:", CODE_ROOT)
if (CODE_ROOT / ".git").exists():
    print("Code commit:", subprocess.check_output(
        ["git", "-C", str(CODE_ROOT), "rev-parse", "HEAD"], text=True,
    ).strip())
# For small code changes, reload the affected module and rerun its caller cells only.


In [ ]:
import csv
import hashlib
import json
import platform
import random
from pathlib import Path

import pandas as pd
import numpy as np
import torch
import torchvision
from torch.utils.data import Subset
from functools import partial
from IPython.display import display

from boundary_unlearning import cifar100, data, models, training, unlearning, phases
from boundary_unlearning.reporting import results_table

# ============================================================
# 1. EXPERIMENT CONFIGURATION
# ============================================================
DATASET_NAME = "CIFAR-100"
DATASET_SLUG = "alincijov/cifar-100"
LABEL_TYPE = "fine"
RUN_PROFILE = "paper"  # "smoke", "paper", "repo_cli"

EXPERIMENT_PHASE = 1
METHODS_TO_RUN = ["boundary_shrink", "boundary_expanding"]
BN_STATS_MODE = "update"  # Author-code behavior; freeze belongs to Phase 2.

FORGET_CLASS = 4
SEED = 2022
BATCH_SIZE = 64
NUM_WORKERS = 2

REUSE_CHECKPOINTS = False
ATTACK_NORMALIZATION = "matched"  # Use the same normalization as the CIFAR-100 input.

FGSM_BOUND = 0.1
UNLEARN_LR = 1e-5
MOMENTUM = 0.9
NUM_CLASSES = 100
EXPECTED_TRAIN_IMAGES = 50_000
EXPECTED_TEST_IMAGES = 10_000
EXTRA_EXP = None  # Phase 1 uses plain Boundary Shrink.

# ============================================================
# 2. TRAINING CONFIGURATION
# ============================================================
if RUN_PROFILE == "smoke":
    ORIGINAL_EPOCHS = 1
    ORIGINAL_LR = 0.1
    UNLEARN_EPOCHS = 1
    MAX_BATCHES_PER_EPOCH = 3
    EVALUATE_TRAIN_SPLITS = False

elif RUN_PROFILE == "paper":
    ORIGINAL_EPOCHS = 30
    ORIGINAL_LR = 0.1
    UNLEARN_EPOCHS = 10
    MAX_BATCHES_PER_EPOCH = None
    EVALUATE_TRAIN_SPLITS = True

elif RUN_PROFILE == "repo_cli":
    ORIGINAL_EPOCHS = 50
    ORIGINAL_LR = 0.01
    UNLEARN_EPOCHS = 10
    MAX_BATCHES_PER_EPOCH = None
    EVALUATE_TRAIN_SPLITS = True

else:
    raise ValueError(
        "RUN_PROFILE must be smoke, paper, or repo_cli"
    )

if ATTACK_NORMALIZATION not in {"matched", "repo_exact"}:
    raise ValueError("Unsupported ATTACK_NORMALIZATION")

if EXTRA_EXP is not None or METHODS_TO_RUN != ["boundary_shrink", "boundary_expanding"]:
    raise ValueError("Phase 1 runs only plain Shrink and Expanding")

if not 0 <= FORGET_CLASS < NUM_CLASSES:
    raise ValueError(f"FORGET_CLASS must be between 0 and {NUM_CLASSES - 1}")

unlearning.validate_methods(METHODS_TO_RUN, extra_exp=EXTRA_EXP)

# ============================================================
# 3. KAGGLE PATHS
# ============================================================
INPUT_ROOT = Path("/kaggle/input")
DATA_ROOT = Path(
    "/kaggle/input/datasets/alincijov/cifar-100"
)

# Kaggle also exposes older dataset mounts without the owner path.
if not DATA_ROOT.is_dir() and Path("/kaggle/input/cifar-100").is_dir():
    DATA_ROOT = Path("/kaggle/input/cifar-100")

OUTPUT_ROOT = Path("/kaggle/working")
RESULT_ROOT = OUTPUT_ROOT / "result"
CHECKPOINT_ROOT = RESULT_ROOT / "checkpoints"

# Kaggle Input is read-only; create directories only in working.
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)
RESULT_ROOT.mkdir(parents=True, exist_ok=True)

# ============================================================
# 4. NORMALIZATION
# ============================================================
CIFAR_MEAN = (0.5, 0.5, 0.5)
CIFAR_STD = (0.5, 0.5, 0.5)

# Used only if ATTACK_NORMALIZATION is changed to "repo_exact".
REPO_ATTACK_MEAN = (0.4914, 0.4822, 0.2265)
REPO_ATTACK_STD = (0.2023, 0.1994, 0.2010)

# ============================================================
# 5. REPRODUCIBILITY
# ============================================================
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# ============================================================
# 6. ENVIRONMENT CHECK
# ============================================================
print("=" * 60)
print("ENVIRONMENT INFORMATION")
print("=" * 60)
print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("NumPy:", np.__version__)
print("Device:", DEVICE)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: GPU is unavailable. Enable GPU in Kaggle settings.")

print("\nInput directories:")
for path in INPUT_ROOT.iterdir():
    print(" -", path)

print("\nData root exists:", DATA_ROOT.exists())
print("Output root:", OUTPUT_ROOT)
print("Checkpoint root:", CHECKPOINT_ROOT)
print("Result root:", RESULT_ROOT)
print("\nConfiguration initialized successfully.")


In [ ]:
# Load the attached Kaggle pickles, including the labeled test split.
DATA_ROOT = cifar100.resolve_data_root(DATA_ROOT)
train_dataset, evaluation_dataset = cifar100.load_cifar100(
    DATA_ROOT, mean=CIFAR_MEAN, std=CIFAR_STD,
)
CIFAR_CLASSES = train_dataset.classes
train_labels = train_dataset.targets
test_labels = evaluation_dataset.targets

if len(train_dataset) != EXPECTED_TRAIN_IMAGES or len(evaluation_dataset) != EXPECTED_TEST_IMAGES:
    raise ValueError("Expected the complete CIFAR-100 train/test splits (50,000 / 10,000)")
train_class_counts = np.bincount(train_labels, minlength=NUM_CLASSES)
test_class_counts = np.bincount(test_labels, minlength=NUM_CLASSES)
if not np.all(train_class_counts == EXPECTED_TRAIN_IMAGES // NUM_CLASSES):
    raise ValueError("Training fine-label counts do not match the balanced CIFAR-100 split")
if not np.all(test_class_counts == EXPECTED_TEST_IMAGES // NUM_CLASSES):
    raise ValueError("Test fine-label counts do not match the balanced CIFAR-100 split")

display(pd.DataFrame([
    {"Split": "Train", "Images": len(train_dataset), "Classes": NUM_CLASSES,
     "Images per class": int(train_class_counts[FORGET_CLASS])},
    {"Split": "Test", "Images": len(evaluation_dataset), "Classes": NUM_CLASSES,
     "Images per class": int(test_class_counts[FORGET_CLASS])},
]).set_index("Split"))
print(f"Forget class: {FORGET_CLASS} ({CIFAR_CLASSES[FORGET_CLASS]})")

# Bind Phase 2 to these exact ordered images and labels.
phase1_data_identity = phases.dataset_identity(train_dataset, evaluation_dataset)


In [ ]:
train_fit_set = train_dataset
train_forget_indices = np.flatnonzero(train_labels == FORGET_CLASS).tolist()
train_retain_indices = np.flatnonzero(train_labels != FORGET_CLASS).tolist()
test_labels = np.asarray(evaluation_dataset.targets, dtype=np.int64)
test_forget_indices = np.flatnonzero(test_labels == FORGET_CLASS).tolist()
test_retain_indices = np.flatnonzero(test_labels != FORGET_CLASS).tolist()

train_forget_set = Subset(train_dataset, train_forget_indices)
train_retain_set = Subset(train_dataset, train_retain_indices)
test_forget_set = Subset(evaluation_dataset, test_forget_indices)
test_retain_set = Subset(evaluation_dataset, test_retain_indices)

print("Train fit:", len(train_fit_set))
print("Train forget:", len(train_forget_set))
print("Train retain:", len(train_retain_set))
print("Official test:", len(evaluation_dataset))
print("Official test forget:", len(test_forget_set))
print("Official test retain:", len(test_retain_set))


In [ ]:
PIN_MEMORY = DEVICE.type == "cuda"
make_loader = partial(
    data.make_loader, batch_size=BATCH_SIZE, seed=SEED,
    num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
)

train_all_loader = make_loader(train_fit_set, shuffle=True)
train_retain_loader = make_loader(train_retain_set, shuffle=True)
train_retain_eval_loader = make_loader(train_retain_set, shuffle=False)
train_forget_eval_loader = make_loader(train_forget_set, shuffle=False)

test_loader = make_loader(evaluation_dataset, shuffle=False)
test_forget_loader = make_loader(test_forget_set, shuffle=False)
test_retain_loader = make_loader(test_retain_set, shuffle=False)

class_count = np.bincount(train_labels, minlength=NUM_CLASSES)

print(f"Fine classes: {len(CIFAR_CLASSES)} | Forget samples per epoch: {len(train_forget_set)}")
print("Train all batches:", len(train_all_loader))
print("Train retain batches:", len(train_retain_loader))
print("Train retain evaluation batches:", len(train_retain_eval_loader))
print("Train forget evaluation batches:", len(train_forget_eval_loader))
print("Official test batches:", len(test_loader))
print("Official test forget batches:", len(test_forget_loader))
print("Official test retain batches:", len(test_retain_loader))

forget_batches = len(make_loader(train_forget_set))
if MAX_BATCHES_PER_EPOCH is not None:
    forget_batches = min(forget_batches, MAX_BATCHES_PER_EPOCH)
print("Unlearning steps per method:", UNLEARN_EPOCHS * forget_batches)


In [ ]:
# ============================================================
# MODEL ARCHITECTURE
# ============================================================

shape_model = models.AllCNN(num_classes=NUM_CLASSES).to(DEVICE)
shape_model.apply(models.init_linear_like_repo)
shape_model.eval()

with torch.no_grad():
    sample_images = torch.randn(2, 3, 32, 32, device=DEVICE)
    sample_logits = shape_model(sample_images)

print("Device:", DEVICE)
print("Model:", shape_model.__class__.__name__)
print("Sample input shape:", sample_images.shape)
print("Sample logits shape:", sample_logits.shape)
print("Total parameters:", sum(p.numel() for p in shape_model.parameters()))
assert sample_logits.shape == (2, NUM_CLASSES)

del shape_model, sample_images, sample_logits


In [ ]:
create_or_load_model = partial(
    training.create_or_load_model, device=DEVICE,
    reuse_checkpoints=REUSE_CHECKPOINTS, num_classes=NUM_CLASSES,
    epochs=ORIGINAL_EPOCHS, learning_rate=ORIGINAL_LR, momentum=MOMENTUM,
    max_batches_per_epoch=MAX_BATCHES_PER_EPOCH,
)


In [ ]:
baseline_tag_settings = {
    "dataset": DATASET_NAME,
    "dataset_slug": DATASET_SLUG,
    "num_classes": NUM_CLASSES,
    "label_type": LABEL_TYPE,
    "input_mean": CIFAR_MEAN,
    "input_std": CIFAR_STD,
    "attack_mean": CIFAR_MEAN if ATTACK_NORMALIZATION == "matched" else REPO_ATTACK_MEAN,
    "attack_std": CIFAR_STD if ATTACK_NORMALIZATION == "matched" else REPO_ATTACK_STD,
    "profile": RUN_PROFILE,
    "data_root": str(DATA_ROOT),
    "implementation_version": "cifar100_v1",
    "evaluation_dataset": "kaggle_cifar100_test_fine",
    "checkpoint_selection": "best_test_accuracy",
    "extra_exp": EXTRA_EXP,
    "method": ["boundary_shrink", "boundary_expanding"],
    "forget_class": FORGET_CLASS,
    "seed": SEED,
    "batch_size": BATCH_SIZE,
    "original_epochs": ORIGINAL_EPOCHS,
    "original_lr": ORIGINAL_LR,
    "unlearn_epochs": UNLEARN_EPOCHS,
    "unlearn_lr": UNLEARN_LR,
    "momentum": MOMENTUM,
    "fgsm_bound": FGSM_BOUND,
    "attack_normalization": ATTACK_NORMALIZATION,
    "max_batches_per_epoch": MAX_BATCHES_PER_EPOCH,
}

baseline_hash = hashlib.sha256(
    json.dumps(baseline_tag_settings, sort_keys=True).encode("utf-8")
).hexdigest()[:8]

baseline_tag = f"cifar100_{RUN_PROFILE}_forget{FORGET_CLASS}_{baseline_hash}"

original_checkpoint = CHECKPOINT_ROOT / f"original_{baseline_tag}.pt"
retrain_checkpoint = CHECKPOINT_ROOT / f"retrain_{baseline_tag}.pt"

print("Baseline tag:", baseline_tag)
print("Original checkpoint:", original_checkpoint)
print("Retrain checkpoint:", retrain_checkpoint)

# Phase identity includes paper-method settings only.
tag_settings = {**baseline_tag_settings, "experiment_phase": 1, "bn_stats_mode": BN_STATS_MODE,
                "evaluated_checkpoint_policy": "snapshot_in_memory_after_evaluation"}
tag_hash = hashlib.sha256(json.dumps(tag_settings, sort_keys=True).encode()).hexdigest()[:8]
run_tag = f"cifar100_phase1_{RUN_PROFILE}_forget{FORGET_CLASS}_{tag_hash}"
PHASE1_BUNDLE_ROOT = RESULT_ROOT / "phase1" / run_tag
print("Phase 1 run tag:", run_tag)
print("Phase 1 bundle:", PHASE1_BUNDLE_ROOT)


In [ ]:
original_model, original_train_seconds = create_or_load_model(
    "original",
    train_all_loader,
    original_checkpoint,
    test_loader,
)


In [ ]:
retrain_model, retrain_train_seconds = create_or_load_model(
    "retrain",
    train_retain_loader,
    retrain_checkpoint,
    test_retain_loader,
)

print("Original model ready:", original_model is not None)
print("Retrain model ready:", retrain_model is not None)
print("Original training time:", original_train_seconds)
print("Retrain training time:", retrain_train_seconds)


## Compare the two paper methods

Run each method independently, then evaluate all four models.


In [ ]:
# Incremental execution: runners only; each following cell trains one paper method.
importlib.reload(unlearning)
if METHODS_TO_RUN != ["boundary_shrink", "boundary_expanding"] or EXTRA_EXP is not None:
    raise ValueError("Phase 1 only supports the two plain paper methods")
run_boundary_shrink = partial(
    unlearning.run_boundary_shrink, device=DEVICE, epochs=UNLEARN_EPOCHS,
    learning_rate=UNLEARN_LR, momentum=MOMENTUM, pin_memory=PIN_MEMORY,
    max_batches_per_epoch=MAX_BATCHES_PER_EPOCH, bound=FGSM_BOUND,
    normalization_mode=ATTACK_NORMALIZATION, extra_exp=None, bn_stats_mode="update",
    cifar_mean=CIFAR_MEAN, cifar_std=CIFAR_STD,
    attack_mean=REPO_ATTACK_MEAN, attack_std=REPO_ATTACK_STD,
)
run_boundary_expanding = partial(
    unlearning.run_boundary_expanding, device=DEVICE, epochs=UNLEARN_EPOCHS,
    learning_rate=UNLEARN_LR, momentum=MOMENTUM, num_classes=NUM_CLASSES,
    pin_memory=PIN_MEMORY, max_batches_per_epoch=MAX_BATCHES_PER_EPOCH,
)
unlearned_models = {name: model for name, model in globals().get("unlearned_models", {}).items()
                    if name in METHODS_TO_RUN}
unlearn_stats = {name: stats for name, stats in globals().get("unlearn_stats", {}).items()
                if name in METHODS_TO_RUN}

def run_method(method_name):
    if method_name not in METHODS_TO_RUN:
        raise ValueError(f"Not a Phase 1 paper method: {method_name}")
    checkpoint = CHECKPOINT_ROOT / f"{method_name}_{run_tag}.pt"
    runner = run_boundary_shrink if method_name == "boundary_shrink" else run_boundary_expanding
    model, stats = runner(original_model, make_loader(train_forget_set, shuffle=True), checkpoint)
    unlearned_models[method_name], unlearn_stats[method_name] = model, stats
    print(f"Ready: {method_name}")


In [ ]:
run_method("boundary_shrink")


In [ ]:
run_method("boundary_expanding")


In [ ]:
from boundary_unlearning.reporting import results_table, unlearning_details_table

evaluate_model_splits = partial(
    training.evaluate_model_splits, device=DEVICE,
    test_loader=test_loader, test_retain_loader=test_retain_loader,
    test_forget_loader=test_forget_loader,
    train_retain_eval_loader=train_retain_eval_loader,
    train_forget_eval_loader=train_forget_eval_loader,
)

# ============================================================
# CIFAR-100 FINE-LABEL TEST EVALUATION
# ============================================================

missing_methods = [name for name in METHODS_TO_RUN if name not in unlearned_models]
if missing_methods:
    raise RuntimeError(f"Run the missing method cells before final evaluation: {missing_methods}")

models_to_evaluate = {
    "original": original_model,
    "retrain": retrain_model,
    **{name: unlearned_models[name] for name in METHODS_TO_RUN},
}

rows = []
baseline_train_times = {"original": original_train_seconds, "retrain": retrain_train_seconds}

for name, model in models_to_evaluate.items():
    row = {"model": name}
    row.update(evaluate_model_splits(model, include_train=EVALUATE_TRAIN_SPLITS))

    if name in baseline_train_times:
        row["training_seconds"] = baseline_train_times[name]
    else:
        row["training_seconds"] = None

    row["unlearning_seconds"] = unlearn_stats[name]["seconds"] if name in unlearn_stats else None
    row["fgsm_cross_rate"] = unlearn_stats[name].get("cross_rate") if name in unlearn_stats else None
    row["unlearning_update_steps"] = unlearn_stats[name].get("update_steps") if name in unlearn_stats else None

    stats = unlearn_stats.get(name, {})
    row["total_unlearning_seconds"] = stats.get("total_seconds", stats.get("seconds"))
    row["student_update_steps"] = stats.get("student_update_steps", stats.get("update_steps"))
    row["total_update_steps"] = stats.get("total_update_steps", stats.get("update_steps"))
    row["mean_sample_weight"] = stats.get("mean_sample_weight")
    row["mean_boundary_epsilon"] = stats.get("mean_boundary_epsilon")
    row["initializer"] = stats.get("initializer", "original" if stats else None)
    row["retain_teacher"] = stats.get("retain_teacher")

    rows.append(row)

display(results_table(rows, dataset_name=DATASET_NAME))
display(unlearning_details_table(rows))

metrics_path = RESULT_ROOT / f"{run_tag}_metrics.csv"
with metrics_path.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
    writer.writeheader()
    writer.writerows(rows)

config = {
    "experiment_phase": EXPERIMENT_PHASE,
    "dataset": DATASET_NAME, "dataset_slug": DATASET_SLUG,
    "dataset_url": f"https://www.kaggle.com/datasets/{DATASET_SLUG}",
    "num_classes": NUM_CLASSES, "label_type": LABEL_TYPE, "class_names": list(CIFAR_CLASSES),
    "model_architecture": {"n_channels": 3, "num_classes": NUM_CLASSES,
                           "filters_percentage": 0.5, "batch_norm": True},
    "input_mean": CIFAR_MEAN, "input_std": CIFAR_STD,
    "attack_mean": CIFAR_MEAN if ATTACK_NORMALIZATION == "matched" else REPO_ATTACK_MEAN,
    "attack_std": CIFAR_STD if ATTACK_NORMALIZATION == "matched" else REPO_ATTACK_STD,
    "run_tag_settings": tag_settings, "baseline_tag": baseline_tag,
    "baseline_tag_settings": baseline_tag_settings,
    "baseline_checkpoints": {"original": str(original_checkpoint), "retrain": str(retrain_checkpoint)},
    "baseline_checkpoint_policy": "best_test_accuracy; distinct from evaluated snapshots",
    "unlearning_stats": {name: unlearn_stats[name] for name in METHODS_TO_RUN},
    "run_profile": RUN_PROFILE, "data_root": str(DATA_ROOT), "methods": METHODS_TO_RUN,
    "forget_class": FORGET_CLASS, "forget_class_name": CIFAR_CLASSES[FORGET_CLASS],
    "seed": SEED, "batch_size": BATCH_SIZE, "num_workers": NUM_WORKERS,
    "evaluation_dataset": "kaggle_cifar100_test_fine",
    "train_sample_count": len(train_fit_set), "test_sample_count": len(evaluation_dataset),
    "extra_exp": EXTRA_EXP, "bn_stats_mode": BN_STATS_MODE,
    "original_epochs": ORIGINAL_EPOCHS, "original_learning_rate": ORIGINAL_LR,
    "unlearn_epochs": UNLEARN_EPOCHS, "unlearn_learning_rate": UNLEARN_LR,
    "momentum": MOMENTUM, "fgsm_bound": FGSM_BOUND,
    "attack_normalization": ATTACK_NORMALIZATION,
    "max_batches_per_epoch": MAX_BATCHES_PER_EPOCH, "evaluate_train_splits": EVALUATE_TRAIN_SPLITS,
    "train_data_shape": list(train_dataset.data.shape), "test_data_shape": list(evaluation_dataset.data.shape),
    "device": str(DEVICE),
    "code_commit": subprocess.check_output(["git", "-C", str(CODE_ROOT), "rev-parse", "HEAD"], text=True).strip()
                   if (Path(CODE_ROOT) / ".git").exists() else None,
    "environment": {"python": platform.python_version(), "torch": torch.__version__,
                    "torchvision": torchvision.__version__, "numpy": np.__version__},
}
config["data_identity"] = phase1_data_identity
config_path = RESULT_ROOT / f"{run_tag}_config.json"
config_path.write_text(json.dumps(config, indent=2), encoding="utf-8")
print("Metrics:", metrics_path)
print("Configuration:", config_path)
# Snapshot immediately: Phase 2 must use these evaluated states, not best-epoch files.
phase1_manifest = phases.save_phase1_bundle(
    models_to_evaluate, rows, config, PHASE1_BUNDLE_ROOT, run_tag=run_tag,
)
print("Phase 1 handoff manifest:", phase1_manifest)


In [ ]:
# Small code update: uncomment these two commands to fetch code from GitHub.
# !git -C /kaggle/working/boundary-unlearning fetch --depth 1 origin main
# !git -C /kaggle/working/boundary-unlearning checkout --detach FETCH_HEAD

import importlib
from boundary_unlearning import attacks, enhanced, unlearning, reporting, artifacts, visualization, phases
importlib.invalidate_caches()
for module in (attacks, enhanced, unlearning, reporting, artifacts, visualization, phases):
    importlib.reload(module)
print("For plot-only changes: rerun Visualization setup and affected plot cells, then export.")
print("Modules reloaded. Rerun runner setup, the affected method cell, evaluation, then export.")
print("Existing Original/Retrain models remain in memory; their training cells are unnecessary.")


## Visualize the four-model comparison

These cells use existing models and evaluation rows; they never train or reload checkpoints.
Run **Visualization setup** after changing models or settings, then any plot cell independently.
The summary plots need only `rows`; Grad-CAM/entropy/t-SNE use the selected model objects.

Sampling is deterministic and shared across models. Grad-CAM pages show the forgotten class
with prediction confidence and P(forget); each heatmap has its own intensity normalization.
Entropy is a descriptive diagnostic. t-SNE panels use separate embeddings with unaligned axes;
CIFAR-100 retain classes share a gray color. Neither plot alone establishes successful unlearning.

Outputs: `result/figures/<run_tag>/` with PNG/PDF, metadata JSON and raw arrays (NPZ).
The final export cell includes every generated figure. t-SNE can be skipped when time is limited.


In [ ]:
# VISUALIZATION SETUP: safe to rerun with existing models; no training.
import importlib
import json
from pathlib import Path
import matplotlib.pyplot as plt
from boundary_unlearning import visualization
importlib.reload(visualization)

VIS_DATASET_NAME = "CIFAR-100" if NUM_CLASSES == 100 else "CIFAR-10"
FIGURE_ROOT = Path(OUTPUT_ROOT) / "result" / "figures" / run_tag
VIS_SEED = SEED
VIS_BATCH_SIZE = 128
VIS_GRADCAM_SAMPLES = 4
VIS_ENTROPY_SAMPLES = 2000  # Per split, bounded before inference.
VIS_TSNE_SAMPLES = 1500    # Total across all classes, including CIFAR-100.
VIS_TSNE_PER_CLASS = 120
VIS_TSNE_PERPLEXITY = 30
VIS_TSNE_ITERATIONS = 1000

missing = [name for name in METHODS_TO_RUN if name not in unlearned_models]
if missing:
    raise RuntimeError(f"Run missing method cells first: {missing}")
# Read model objects again when setup is rerun, avoiding stale model references.
visualization_models = {
    "original": original_model,
    "retrain": retrain_model,
    **{name: unlearned_models[name] for name in METHODS_TO_RUN},
}
visualization_datasets = {
    "train_retain": train_retain_set,
    "train_forget": train_forget_set,
    "test": evaluation_dataset,
}
gradcam_indices = visualization.select_sample_indices(
    train_forget_set, max_samples=VIS_GRADCAM_SAMPLES, seed=VIS_SEED,
)
entropy_indices = {
    split: visualization.select_sample_indices(dataset, max_samples=VIS_ENTROPY_SAMPLES, seed=VIS_SEED)
    for split, dataset in visualization_datasets.items()
}
tsne_indices = visualization.select_sample_indices(
    evaluation_dataset, max_samples=VIS_TSNE_SAMPLES, per_class=VIS_TSNE_PER_CLASS, seed=VIS_SEED,
)
FIGURE_ROOT.mkdir(parents=True, exist_ok=True)
visualization_settings = {
    "run_tag": run_tag, "dataset": VIS_DATASET_NAME, "seed": VIS_SEED,
    "models": list(visualization_models), "class_names": list(CIFAR_CLASSES),
    "forget_class": FORGET_CLASS, "input_mean": CIFAR_MEAN, "input_std": CIFAR_STD,
    "gradcam_split": "train_forget", "gradcam_indices": gradcam_indices.tolist(),
    "entropy_indices": {split: indices.tolist() for split, indices in entropy_indices.items()},
    "tsne_split": "test", "tsne_indices": tsne_indices.tolist(),
    "tsne_perplexity": VIS_TSNE_PERPLEXITY, "tsne_max_iter": VIS_TSNE_ITERATIONS,
    "entropy_sample_limit": VIS_ENTROPY_SAMPLES, "tsne_sample_limit": VIS_TSNE_SAMPLES,
    "tsne_per_class_limit": VIS_TSNE_PER_CLASS, "inference_batch_size": VIS_BATCH_SIZE,
    "model_source": "current_in_memory_models", "data_root": str(DATA_ROOT),
}
(FIGURE_ROOT / "visualization_config.json").write_text(
    json.dumps(visualization_settings, ensure_ascii=False, indent=2), encoding="utf-8",
)
print("Figure folder:", FIGURE_ROOT)
print("Models:", len(visualization_models), "| t-SNE samples:", len(tsne_indices))


In [ ]:
# VISUALIZATION: accuracy and cost. Can run without model objects or visualization setup.
from pathlib import Path
import importlib
import matplotlib.pyplot as plt
from boundary_unlearning import visualization
importlib.reload(visualization)
FIGURE_ROOT = Path(OUTPUT_ROOT) / "result" / "figures" / run_tag
VIS_DATASET_NAME = "CIFAR-100" if NUM_CLASSES == 100 else "CIFAR-10"
fig = visualization.plot_metric_comparison(rows, output_dir=FIGURE_ROOT, dataset_name=VIS_DATASET_NAME)
plt.show()
plt.close(fig)
fig = visualization.plot_cost_comparison(rows, output_dir=FIGURE_ROOT, dataset_name=VIS_DATASET_NAME)
plt.show()
plt.close(fig)


In [ ]:
# VISUALIZATION: Grad-CAM on fixed forget samples; no weight updates.
for fig in visualization.plot_gradcam_comparison(
    visualization_models, train_forget_set, gradcam_indices,
    device=DEVICE, class_names=CIFAR_CLASSES, forget_class=FORGET_CLASS,
    mean=CIFAR_MEAN, std=CIFAR_STD, output_dir=FIGURE_ROOT,
    dataset_name=VIS_DATASET_NAME, models_per_page=4,
):
    plt.show()
    plt.close(fig)


In [ ]:
# VISUALIZATION: entropy distributions on the same selected samples per model.
fig = visualization.plot_entropy_comparison(
    visualization_models, visualization_datasets, entropy_indices,
    device=DEVICE, batch_size=VIS_BATCH_SIZE,
    output_dir=FIGURE_ROOT, dataset_name=VIS_DATASET_NAME,
)
plt.show()
plt.close(fig)


In [ ]:
# VISUALIZATION: optional t-SNE; skip this cell for a quick export.
fig = visualization.plot_tsne_comparison(
    visualization_models, evaluation_dataset, tsne_indices,
    device=DEVICE, class_names=CIFAR_CLASSES, forget_class=FORGET_CLASS,
    seed=VIS_SEED, perplexity=VIS_TSNE_PERPLEXITY, max_iter=VIS_TSNE_ITERATIONS,
    batch_size=VIS_BATCH_SIZE, output_dir=FIGURE_ROOT, dataset_name=VIS_DATASET_NAME,
)
plt.show()
plt.close(fig)


## Optional: Phase 1 epoch comparison

After evaluation, run this cell separately to compare unlearning budgets from the exact same exported Original. It never retrains Original/Retrain or overwrites the main four-model results. Each budget has its own complete Phase 1 bundle for Phase 2.


In [ ]:
# EPOCH COMPARISON: opt in after Phase 1 evaluation; no Run All required.
RUN_EPOCH_SWEEP = False
UNLEARN_EPOCHS_TO_TEST = [1, 3, 5, 10, 20]
EPOCH_SWEEP_SOURCE = globals().get("phase1_manifest")  # Or a Path to an attached Phase 1 manifest/folder/ZIP.

if RUN_EPOCH_SWEEP:
    exec((Path(CODE_ROOT) / "experiments" / "run-epoch-sweep.py").read_text())
else:
    print("Epoch comparison skipped. Enable RUN_EPOCH_SWEEP to run this cell separately.")


## Export Phase 1 and download

Download `result.zip`; attach it or the saved notebook Output to Phase 2.


In [ ]:
# ============================================================
# EXPORT RESULTS: run this cell independently to refresh the ZIP.
# ============================================================
from pathlib import Path
import os
from IPython.display import FileLink, display
from boundary_unlearning.artifacts import export_results
from boundary_unlearning.reporting import results_table, unlearning_details_table

# Use the singular folder name even in a session using the older path settings.
result_dir = Path(OUTPUT_ROOT) / "result"
result_dir.mkdir(parents=True, exist_ok=True)
table_path = result_dir / f"{run_tag}_results.html"
results_table(rows, dataset_name=DATASET_NAME).to_html(table_path)

details_path = result_dir / f"{run_tag}_unlearning_details.html"
unlearning_details_table(rows).to_html(details_path)

# Includes existing result/figures/ recursively; no plot is recomputed here.
if not Path(phase1_manifest).is_file():
    raise FileNotFoundError("Run Phase 1 evaluation before exporting")
archive_path = export_results(
    result_dir,
    checkpoint_dir=CHECKPOINT_ROOT,
    files=[metrics_path, config_path],
)
print(f"Archive: {archive_path} ({archive_path.stat().st_size / (1024 ** 2):.2f} MiB)")
display(FileLink(os.path.relpath(archive_path), result_html_prefix="Download: "))

print("Phase 2 input: upload this ZIP as a Kaggle Dataset, or attach the saved Phase 1 notebook Output.")
print("Evaluated-model manifest:", phase1_manifest)
